# Candidate SSE Socio-Geodemographic Association

Lightweight runner for the main overall association analysis. See [`sse_sociodemographic_association_notes.md`](sse_sociodemographic_association_notes.md) for the model rationale, inputs, output table definitions, and interpretation guide.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "config.yaml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from sse_detection import lib as sselib  # noqa: E402

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 220)

## Configuration

In [ ]:
RESULT_DIR = PROJECT_ROOT / "sse_detection" / "association_outputs"
MODEL_METHOD = "firth_glm"
VARIANT_ADJUSTER = "clade"
WINDOW_ADJUSTMENT = "fixed_effects"
MIXING_REFERENCE = "per 1 null-model SD increase in entropy"

model_sets = sselib.default_model_sets(
    variant_adjuster=VARIANT_ADJUSTER,
    window_adjustment=WINDOW_ADJUSTMENT,
)
model_sets

## Fit and Save

In [ ]:
result = sselib.run_main_association_analysis(
    project_root=PROJECT_ROOT,
    result_dir=RESULT_DIR,
    model_method=MODEL_METHOD,
    variant_adjuster=VARIANT_ADJUSTER,
    window_adjustment=WINDOW_ADJUSTMENT,
    mixing_reference=MIXING_REFERENCE,
)

summary_tables = result["summary_tables"]
composition_wald = summary_tables["composition_wald.csv"]
composition_or = summary_tables["composition_odds_ratios.csv"]
composition_fit_stats = summary_tables["composition_fit_stats.csv"]
mixing_wald = summary_tables["mixing_wald.csv"]
mixing_or = summary_tables["mixing_odds_ratios.csv"]
mixing_fit_stats = summary_tables["mixing_fit_stats.csv"]

print(f"Results saved to: {result['result_dir']}")
{name: len(table) for name, table in summary_tables.items()}

## Frame Diagnostics

In [ ]:
frames = result["frames"]

display(
    pd.DataFrame(
        [
            {
                "frame": "node_stats",
                "rows": len(frames.node_stats),
                "nodes": frames.node_stats["cluster_id"].nunique(),
                "candidate_rows": int(frames.node_stats["sse_candidate"].sum()),
            },
            {
                "frame": "eligible_nodes",
                "rows": len(frames.eligible_nodes),
                "nodes": frames.eligible_nodes["cluster_id"].nunique(),
                "candidate_rows": int(frames.eligible_nodes["sse_candidate"].sum()),
            },
            {
                "frame": "composition_base",
                "rows": len(frames.composition_base),
                "nodes": frames.composition_base["cluster_id"].nunique(),
                "candidate_rows": int(frames.composition_base["candidate"].sum()),
            },
        ]
    )
)

print(f"Minimum candidate cluster size: {frames.min_candidate_size}")
display(result["cluster_diagnostics"])

## Composition Tables

In [ ]:
display(sselib.select_table_columns(composition_wald, "wald"))
display(sselib.select_table_columns(composition_fit_stats, "fit_stats"))

In [ ]:
display(
    sselib.select_table_columns(
        composition_or,
        "odds_ratios",
        sort_by=["model_set", "predictor_set", "predictor", "p_value"],
    )
)

## Mixing Tables

In [ ]:
display(sselib.select_table_columns(mixing_wald, "wald"))
display(sselib.select_table_columns(mixing_fit_stats, "fit_stats"))

In [ ]:
display(
    sselib.select_table_columns(
        mixing_or,
        "odds_ratios",
        sort_by=["model_set", "predictor_set", "predictor", "p_value"],
    )
)

## Fit Failures

In [ ]:
if not result["failures"].empty:
    display(result["failures"])
else:
    print("No model failures recorded.")